# 02 - Results analysis: raw frames versus pose keypoints

Run after `evaluate.py`, `benchmark_speed.py` and `compare.py` (see `notebooks/colab/m4_evaluation_demo.ipynb`).

In [ ]:
import json
import sys
from itertools import combinations
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, Markdown, display
from scipy.stats import binomtest

from src.utils.config import load_config, resolve_paths

RESULTS = resolve_paths(load_config(ROOT / "configs" / "data.yaml"))["results_root"]
COMP = RESULTS / "comparison"

## The comparison table

In [ ]:
display(Markdown((COMP / "comparison.md").read_text(encoding="utf-8")))

## Accuracy against speed
Left: the model alone, starting from cached frames or keypoints. Right (if measured): from a raw video file, which
includes decoding and, for the pose models, MediaPipe.

In [ ]:
display(Image(filename=str(COMP / "accuracy_vs_latency.png")))
display(Image(filename=str(COMP / "accuracy_bars.png")))

## Is a difference between two models real? (paired test)
All models are tested on the same videos, so a paired exact test (McNemar) is appropriate: it only looks at the
videos on which exactly one of the two models is right.

In [ ]:
preds = {d.name: pd.read_csv(d / "eval_test" / "predictions.csv").set_index("video_id")["correct"]
         for d in sorted(RESULTS.iterdir()) if (d / "eval_test" / "predictions.csv").exists()}
rows = []
for a, b in combinations(preds, 2):
    both = pd.concat([preds[a], preds[b]], axis=1, keys=["a", "b"]).dropna().astype(bool)
    only_a, only_b = int((both.a & ~both.b).sum()), int((~both.a & both.b).sum())
    p = binomtest(only_a, only_a + only_b, 0.5).pvalue if only_a + only_b else 1.0
    rows.append({"A": a, "B": b, "only A right": only_a, "only B right": only_b, "p-value": round(p, 3)})
pd.DataFrame(rows)

## Most frequent mistakes per model

In [ ]:
for d in sorted(RESULTS.iterdir()):
    f = d / "eval_test" / "top_confusions.json"
    if f.exists():
        print(d.name)
        display(pd.DataFrame(json.loads(f.read_text())[:6]))

## Weakest classes per model

In [ ]:
for d in sorted(RESULTS.iterdir()):
    f = d / "eval_test" / "per_class.csv"
    if f.exists():
        print(d.name)
        display(pd.read_csv(f).head(5))